# 02 HumAID: fine-tune a transformer classifier

Member 5. Fine-tunes a tweet-trained RoBERTa on the HumAID splits saved by notebook 01 and compares it with the TF-IDF baseline (macro-F1 0.712 validation, 0.714 test).

**Needs a GPU runtime** (Runtime, Change runtime type, T4). Make sure no other notebook is using the GPU.
**Run it twice:** first with `SMOKE = True` (2,000 tweets, one epoch, a few minutes) to check that everything works, then set `SMOKE = False` for the real run. A real run saves the model to Drive. Clear outputs before committing. Never commit the model or the data.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "scikit-learn", "pyarrow"], check=True)

import os, random, copy
import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
DATA = f"{ROOT}/humaid"                       # parquet files saved by notebook 01

MODEL_NAME = "cardiffnlp/twitter-roberta-base"
MAX_LEN = 128
BATCH = 32
EPOCHS = 3
LR = 2e-5
WEIGHT_POWER = 1.0        # 1.0 = balanced class weights (same idea as the baseline); try 0.5 for softer weights
SEED = 42
SMOKE = False              # True: tiny quick run to test the notebook. False: the real run.

BASELINE = {"validation": {"accuracy": 0.732, "macro_f1": 0.712},
            "test": {"accuracy": 0.733, "macro_f1": 0.714}}

BASELINE_ACTIONABLE = 0.739   # mean F1 of the six actionable classes for the baseline on test (from its per-class report)

# classes the priority list cares about; edit once the team has agreed on them
ACTIONABLE = ["infrastructure_and_utility_damage", "injured_or_dead_people", "displaced_people_and_evacuations",
              "requests_or_urgent_needs", "missing_or_found_people", "caution_and_advice"]

OUT = f"{DATA}/transformer" + ("_smoke" if SMOKE else "")
os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| smoke:", SMOKE)
if device == "cpu" and not SMOKE:
    raise SystemExit("No GPU found. Change the runtime type to GPU, or set SMOKE = True.")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## Data

In [ ]:
def load_frames(data_dir, smoke=False, seed=SEED):
    """train, validation and test DataFrames (columns text, label). A smoke run takes a small random subset."""
    frames = {n: pd.read_parquet(f"{data_dir}/{n}.parquet") for n in ("train", "validation", "test")}
    if smoke:
        sizes = {"train": 2000, "validation": 500, "test": 500}
        frames = {n: df.sample(n=min(sizes[n], len(df)), random_state=seed).reset_index(drop=True) for n, df in frames.items()}
    return frames


frames = load_frames(DATA, SMOKE)
LABELS = sorted(pd.read_parquet(f"{DATA}/train.parquet")["label"].unique())     # fixed from the full train split
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print({k: len(v) for k, v in frames.items()}, "| classes:", len(LABELS))

In [ ]:
def class_weights(labels, n_classes, power=WEIGHT_POWER):
    """Weight per class: (n / (k * count)) ** power, so rare classes count more in the loss."""
    counts = np.bincount(labels, minlength=n_classes).clip(min=1)
    w = (len(labels) / (n_classes * counts)) ** power
    return torch.tensor(w, dtype=torch.float32)


def truncated_share(tokenizer, texts, max_len):
    """Share of tweets longer than max_len tokens (these get cut)."""
    lengths = [len(x) for x in tokenizer(list(texts), truncation=False)["input_ids"]]
    return float(np.mean(np.array(lengths) > max_len))


def make_examples(tokenizer, df, max_len):
    """One dict per tweet with input_ids, attention_mask and labels (padding happens per batch)."""
    enc = tokenizer(list(df["text"]), truncation=True, max_length=max_len)
    y = df["label"].map(label2id).tolist()
    return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": y[i]}
            for i in range(len(df))]


def make_loader(tokenizer, examples, batch, shuffle):
    """DataLoader that pads each batch to its longest tweet."""
    def collate(items):
        labels = torch.tensor([x["labels"] for x in items])
        batch_enc = tokenizer.pad([{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in items],
                                  return_tensors="pt")
        batch_enc["labels"] = labels
        return batch_enc
    return torch.utils.data.DataLoader(examples, batch_size=batch, shuffle=shuffle, collate_fn=collate)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("tweets longer than", MAX_LEN, "tokens (train):", round(truncated_share(tokenizer, frames["train"]["text"], MAX_LEN), 4))

examples = {n: make_examples(tokenizer, df, MAX_LEN) for n, df in frames.items()}
loaders = {"train": make_loader(tokenizer, examples["train"], BATCH, True),
           "validation": make_loader(tokenizer, examples["validation"], BATCH * 2, False),
           "test": make_loader(tokenizer, examples["test"], BATCH * 2, False)}
weights = class_weights(frames["train"]["label"].map(label2id).to_numpy(), len(LABELS))
print(pd.Series(weights.numpy(), index=LABELS).round(2))

## Training

In [ ]:
from sklearn.metrics import accuracy_score, f1_score


def predict(model, loader, device):
    """Predicted class ids, class probabilities and true ids for one loader."""
    model.eval()
    preds, probs, ys = [], [], []
    with torch.no_grad():
        for b in loader:
            y = b.pop("labels")
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(**b).logits
            p = torch.softmax(logits.float(), dim=-1).cpu()
            preds.append(p.argmax(-1)); probs.append(p); ys.append(y)
    return torch.cat(preds).numpy(), torch.cat(probs).numpy(), torch.cat(ys).numpy()


def scores(y_true, y_pred):
    """Accuracy and macro-F1."""
    return {"accuracy": round(float(accuracy_score(y_true, y_pred)), 3),
            "macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 3)}


def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler, device):
    """One pass over the training data with mixed precision on GPU. Returns the mean loss."""
    model.train()
    total, n = 0.0, 0
    for b in loader:
        y = b.pop("labels").to(device)
        b = {k: v.to(device) for k, v in b.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item() * len(y); n += len(y)
    return total / n

In [ ]:
def fit(model_name, loaders, weights, epochs, lr, device):
    """Train, keep the epoch with the best validation macro-F1, return the model and a history table."""
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * len(loaders["train"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(device))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    best, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, loaders["train"], optimizer, scheduler, loss_fn, scaler, device)
        pred, _, y = predict(model, loaders["validation"], device)
        s = scores(y, pred)
        history.append({"epoch": epoch, "train_loss": round(loss, 4), **{f"val_{k}": v for k, v in s.items()}})
        print(history[-1])
        if s["macro_f1"] > best:
            best = s["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


model, history = fit(MODEL_NAME, loaders, weights, 1 if SMOKE else EPOCHS, LR, device)
history

## Results

In [ ]:
pred_val, _, y_val = predict(model, loaders["validation"], device)
pred_test, prob_test, y_test = predict(model, loaders["test"], device)

results = pd.DataFrame({"TF-IDF baseline": {f"{s} {m}": BASELINE[s][m] for s in BASELINE for m in ("accuracy", "macro_f1")},
                        "transformer": {**{f"validation {k}": v for k, v in scores(y_val, pred_val).items()},
                                        **{f"test {k}": v for k, v in scores(y_test, pred_test).items()}}})
if SMOKE:
    print("Smoke run: these numbers come from a tiny subset and are not results.")
results

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix


def per_class_report(y_true, y_pred):
    """Precision, recall, F1 and support per class."""
    rep = classification_report(y_true, y_pred, labels=range(len(LABELS)), target_names=LABELS,
                                output_dict=True, zero_division=0)
    return pd.DataFrame(rep).T.round(3)


def group_f1(y_true, y_pred, group):
    """Macro-F1 over the classes in group only (for example the actionable ones)."""
    ids = [label2id[g] for g in group]
    return round(float(f1_score(y_true, y_pred, labels=ids, average="macro")), 3)


def plot_confusion(y_true, y_pred, title):
    """Row-normalised confusion matrix with counts in the cells."""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(LABELS)))
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(LABELS)), LABELS, rotation=60, ha="right")
    ax.set_yticks(range(len(LABELS)), LABELS)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    fig.tight_layout()
    return fig


print(per_class_report(y_test, pred_test))
print("macro-F1 on the actionable classes (test):", group_f1(y_test, pred_test, ACTIONABLE), "| baseline:", BASELINE_ACTIONABLE)
fig = plot_confusion(y_test, pred_test, "Transformer on test")

## Save the model and the test predictions

In [ ]:
def save_all(model, tokenizer, out_dir, texts, y_true, pred, probs):
    """Model, tokenizer and label map for later use, and the test predictions for the retrieval step."""
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    df = pd.DataFrame({"text": list(texts), "label": [id2label[i] for i in y_true],
                       "pred": [id2label[i] for i in pred], "confidence": probs.max(axis=1).round(4)})
    df.to_parquet(f"{out_dir}/test_predictions.parquet", index=False)
    return df


test_preds = save_all(model, tokenizer, OUT, frames["test"]["text"], y_test, pred_test, prob_test)
print("saved to", OUT)
test_preds.head()

In [ ]:
def classify(texts, model, tokenizer, device, max_len=MAX_LEN):
    """Class name and probability for each text. This is the function the demo and retrieval step call."""
    model.eval()
    enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy()
    return [(id2label[int(i)], round(float(p[k, i]), 3)) for k, i in enumerate(p.argmax(1))]


classify(["Roads are flooded and the bridge has collapsed, we need boats",
          "Praying for everyone affected by the floods",
          "Please send water and medicine to the shelter"], model, tokenizer, device)

## Notes for the report
- Report macro-F1 and accuracy for the baseline and the transformer on the same test split, plus the macro-F1 on the actionable classes. Give the support next to the per-class scores: `missing_or_found_people` has only 72 test tweets, so its F1 is noisy.
- Name the model, the learning rate, epochs, batch size, maximum length, class weighting (`WEIGHT_POWER`) and seed, and that the best epoch was picked on the validation split only. The test split was used once.
- HumAID is English only and covers 2016 to 2019 events. Say that scores on new events or on flood tweets only may differ.
- Never commit the model or the data. The Drive folder holds both.